- Before jumping into results - a brief section on the tests & model checks conducted before running regression e.g. detecting multicollinearity with VIF, potential endogeneity, confounding, etc.
- How are controls actually implemented in the OLS model?

- Just for the sake of it, could we also try one mega model using all the predictors?
- Correlation chart between outputs & input features
- It would be good to translate what -ve/+ve interaction terms mean intuitively (for income x race)
- It would be good to also add to the notebook the summary regression tables for the other versions under each model e.g. 3A, 3B etc.

- Can we run similar analysis for other DERs per capita as well (similar to EV chargers)?
    - This could also entail that some predictor variables could be used as controls (e.g. PV per capita as control while predicting storage in the same ZIP code, or fossil fuel plant capacity per capita while predicting PV/wind)
    - The county-level / infra fixed effects may be more significant here for predicting PV/wind capacity per capita (while also controlling for solar radiation/wind speed availability)
    - For wind (or PV) per capita I think we should also add avg annual wind speed (or solar radiation) as a feature (either as predictor or control)

In [166]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import cross_val_score
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.impute import SimpleImputer
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col
import patsy
import geopandas as gpd

In [167]:
base = "../outputs/tables/"
y = "log_chargers_per_10k"  # <-- change this to y_pv, y_storage, etc.

def prep_outcomes_per_capita(df, pop_col="total_population", min_pop=1000):
    """Create per-capita + log1p outcomes; filter tiny-pop ZIPs."""
    df = df.copy()
    df[pop_col] = pd.to_numeric(df[pop_col], errors="coerce")
    df = df[df[pop_col].notna() & (df[pop_col] >= min_pop)].copy()

    pop = df[pop_col]

    df["chargers_per_1k"] = df["total_chargers"] * 1000 / pop
    df["y_chargers"] = np.log1p(df["chargers_per_1k"])

    df["pv_kw_per_1k"] = df["PV_system_size_DC"] * 1000 / pop
    df["y_pv"] = np.log1p(df["pv_kw_per_1k"])

    df["storage_mw_per_100k"] = df["storage_capacity_mw"] * 100000 / pop
    df["y_storage"] = np.log1p(df["storage_mw_per_100k"])

    df["wind_mw_per_100k"] = df["wind_capacity_mw"] * 100000 / pop
    df["y_wind_mw"] = np.log1p(df["wind_mw_per_100k"])

    df["y_turbines"] = np.log1p(df["wind_turbine_count"])
    df["any_turbines"] = (df["wind_turbine_count"] > 0).astype(int)

    return df

def center_cols(df, cols):
    """Mean-center columns for interaction models."""
    df = df.copy()
    for c in cols:
        if c in df.columns:
            df[c + "_c"] = df[c] - df[c].mean()
    return df

def run_ols(formula, df, cluster_col=None):
    """Run OLS with HC1 SEs by default; optional clustered SEs."""
    m = smf.ols(formula=formula, data=df)

    if cluster_col is None:
        return m.fit(cov_type="HC1")
    res = m.fit()
    used_idx = res.model.data.row_labels
    groups = df.loc[used_idx, cluster_col].copy()
    groups = groups.fillna("MISSING").astype(str)
    return m.fit(cov_type="cluster", cov_kwds={"groups": groups})

def quick_print(res, title=""):
    if title:
        print("\n" + "="*80)
        print(title)
        print("="*80)
    print(res.summary().tables[0])
    print(res.summary().tables[1])

    f = open("../outputs/tables/" + title + ".html", 'w')
    f.write(res.summary().as_html())
    res.summary2().tables[1].to_csv(base + title + ".csv")



def vif_from_formula(formula, df, exclude_prefixes=("C(",), drop_intercept=True):
    y, X = patsy.dmatrices(formula, df, return_type="dataframe")

    if drop_intercept and "Intercept" in X.columns:
        X = X.drop(columns=["Intercept"])

    # optionally exclude fixed-effect dummy columns
    if exclude_prefixes:
        keep = []
        for col in X.columns:
            if not any(col.startswith(pref) for pref in exclude_prefixes):
                keep.append(col)
        X = X[keep]

    X = X.replace([np.inf, -np.inf], np.nan).dropna()

    vif = pd.DataFrame({
        "feature": X.columns,
        "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    }).sort_values("VIF", ascending=False)

    return vif

In [168]:
#### THIS IS JUST FOR CALCULATING DIFFERENT FEATURES
#### ONLY MODIFY TO ADD FEATURES
df = pd.read_csv("../data/processed/combined_der_dataset_w_controls_predictors.csv")
df.drop(columns=['Unnamed: 0'], inplace=True)
df.rename(columns={"ghi_mean_kwh_m2_day_2024":"ghi_mean_kwh_m2_day_2023"}, inplace=True)
# numeric coercion for key vars (safe)
for c in [
    "median_household_income","poverty_rate",
    "pct_black","pct_hispanic","pct_asian",
    "cdd65_2023","hdd65_2023","t2m_mean_c_2023",
    "ghi_mean_kwh_m2_day_2023","total_population",
    "lat","lon"
]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# fill utility_type if you plan to use it
if "utility_type" in df.columns:
    df["utility_type"] = df["utility_type"].fillna("POU")

print(df.shape, df.columns[:10])

df = prep_outcomes_per_capita(df, min_pop=1000)

# keep only rows with core predictors present
core_needed = ["median_household_income", "pct_black", "pct_hispanic", "pct_asian"]
df = df.dropna(subset=[c for c in core_needed if c in df.columns]).copy()

# outcomes available
outcomes = [c for c in ["y_chargers", "y_pv", "y_storage", "y_wind_mw", "y_turbines"] if c in df.columns]
outcomes, df.shape

df["chargers_per_10k"] = df["y_chargers"] / df["total_population"] * 10000
df["log_chargers_per_10k"] = np.log1p(df["chargers_per_10k"])
df["log_pop"] = np.log1p(df["total_population"])

ZCTA_SHP = "../data/raw/boundaries/tl_2023_us_zcta520/tl_2023_us_zcta520.shp"
COUNTY_SHP = "../data/raw/boundaries/tl_2023_us_county/tl_2023_us_county.shp"
zcta = gpd.read_file(ZCTA_SHP)
county = gpd.read_file(COUNTY_SHP)
zcta_id = "ZCTA5CE20" if "ZCTA5CE20" in zcta.columns else ("GEOID20" if "GEOID20" in zcta.columns else "GEOID")
county_id = "GEOID" if "GEOID" in county.columns else ("GEOID20" if "GEOID20" in county.columns else "GEOID10")
county_name_col = "NAME" if "NAME" in county.columns else None
zcta = zcta[[zcta_id, "geometry"]].rename(columns={zcta_id: "zip_code"})
keep = [county_id, "geometry"] + ([county_name_col] if county_name_col else [])
county = county[keep].rename(columns={county_id: "county_geoid"})
if county_name_col:
    county = county.rename(columns={county_name_col: "county_name"})
zcta["zip_code"] = zcta["zip_code"].astype(str).str.zfill(5)
zcta = zcta.to_crs("EPSG:5070")
county = county.to_crs("EPSG:5070")
pairs = gpd.sjoin(zcta, county, how="inner", predicate="intersects").drop(columns=["index_right"])
county_geom = county.set_index("county_geoid").geometry
other = gpd.GeoSeries(pairs["county_geoid"].map(county_geom), index=pairs.index, crs=pairs.crs)
pairs["overlap_area"] = pairs.geometry.intersection(other).area
pairs = pairs.sort_values(["zip_code", "overlap_area"], ascending=[True, False])
zip_to_county = pairs.drop_duplicates("zip_code")[["zip_code", "county_geoid"] + (["county_name"] if "county_name" in pairs.columns else [])]
df["zip_code"] = df["zip_code"].astype(str).str.zfill(5)
df = df.merge(zip_to_county, on="zip_code", how="left")
print(df[["zip_code", "county_geoid"] + (["county_name"] if "county_name" in df.columns else [])].head())

zcta = gpd.read_file(ZCTA_SHP)
zcta_id = "ZCTA5CE20" if "ZCTA5CE20" in zcta.columns else "GEOID20"
zcta = zcta[[zcta_id, "geometry"]].rename(columns={zcta_id: "zip_code"})
zcta["zip_code"] = zcta["zip_code"].astype(str).str.zfill(5)
zcta = zcta.to_crs("EPSG:5070")
zcta["area_km2"] = zcta.geometry.area / 1e6
df["zip_code"] = df["zip_code"].astype(str).str.zfill(5)
df = df.merge(zcta[["zip_code", "area_km2"]], on="zip_code", how="left")
df["pop_density_km2"] = df["total_population"] / df["area_km2"].replace(0, np.nan)
df["log_pop_density"] = np.log1p(df["pop_density_km2"])
min_n = 5
df["county_geoid"] = df["county_geoid"].astype(str).str.strip()
counts = df["county_geoid"].value_counts()
keep = counts[counts >= min_n].index
df = df[df["county_geoid"].isin(keep)].copy()

(2586, 38) Index(['Unnamed: 0.1', 'zip_code', 'PV_system_size_DC', 'total_chargers',
       'level1_chargers', 'level2_chargers', 'dc_fast_chargers', 'zev_count',
       'plant_capacity_mw', 'storage_capacity_mw'],
      dtype='object')
  zip_code county_geoid  county_name
0    90001        06037  Los Angeles
1    90002        06037  Los Angeles
2    90003        06037  Los Angeles
3    90004        06037  Los Angeles
4    90005        06037  Los Angeles


In [169]:
#### This is where the variables are defined to be used in the models
income = "median_household_income"
race = ["pct_black", "pct_hispanic", "pct_asian"]

# optional control: include poverty_rate or not
controls_common = ["poverty_rate", "total_population"] # "poverty_rate"

# climate strategies
controls_3A = ["cdd65_2023", "hdd65_2023"]  # degree-days
controls_3B = ["t2m_mean_c_2023"]          # temp-only
controls_3C = ["ghi_mean_kwh_m2_day_2023"] # PV only

# SES proxies
ses_bach = ["pct_bachelors_plus"]
ses_house = ["median_housing_value"]

# FE and clustering columns
utility_fe = "C(utility)"
cluster_utility = "utility"

df["county_geoid"] = df["county_geoid"].astype(str)
county_fe = "C(county_geoid)"

latlon = ["lat","lon"]
lat = ["lat"]
lon = ["lon"]

# optional demand proxy (only if you merged it)
demand_proxy = "log_kwh"
df = df[df["county_geoid"].astype(str).str.startswith("06")].copy()
controls_common, controls_3A, controls_3B, utility_fe

(['poverty_rate', 'total_population'],
 ['cdd65_2023', 'hdd65_2023'],
 ['t2m_mean_c_2023'],
 'C(utility)')

In [170]:
def build_formula(y, climate_controls, extra_terms=None, fe_terms=None,
                  keepincome=True, demand_proxy=None, controls_common = []):
    rhs = []
    if keepincome:
        rhs.append(income)
    rhs += race
    rhs += list(controls_common)
    if demand_proxy is not None:
        rhs.append(demand_proxy)
    rhs += list(climate_controls)
    if extra_terms:
        rhs += list(extra_terms)
    if fe_terms:
        rhs += list(fe_terms)
    seen = set()
    rhs = [x for x in rhs if not (x in seen or seen.add(x))]

    return f"{y} ~ " + " + ".join(rhs)

Model 1 (Baseline, degree-days climate)

In [171]:
f1 = build_formula(
    y,
    climate_controls=controls_3A,
    controls_common=["poverty_rate"],
    fe_terms=[])
res1 = run_ols(f1, df)
title = f"{y} | Model 1 baseline (CDD+HDD)"
quick_print(res1, title)
print(vif_from_formula(f1, df).head(30))


log_chargers_per_10k | Model 1 baseline (CDD+HDD)
                             OLS Regression Results                             
Dep. Variable:     log_chargers_per_10k   R-squared:                       0.203
Model:                              OLS   Adj. R-squared:                  0.198
Method:                   Least Squares   F-statistic:                     24.14
Date:                  Fri, 30 Jan 2026   Prob (F-statistic):           1.25e-30
Time:                          23:16:09   Log-Likelihood:                -832.68
No. Observations:                  1072   AIC:                             1681.
Df Residuals:                      1064   BIC:                             1721.
Df Model:                             7                                         
Covariance Type:                    HC1                                         
                              coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------

**Model 1 (baseline)**
In our baseline specification, the dependent variable is log(1 + chargers per 10,000 residents), which retains ZIP codes with zero chargers and reduces skew. Using OLS with heteroskedasticity-robust (HC1) standard errors, we find that higher Black, Hispanic, and Asian population shares are each associated with lower charger availability, controlling for median household income, poverty rate, and climate demand (HDD/CDD). For example, a 10 percentage point increase in Hispanic share is associated with roughly a 7% decrease in (1 + chargers per 10,000). Median household income is not statistically significant in this baseline.

Model 2 (SES robustness: add ONE proxy)

In [172]:
# bachelors proxy
f2a = build_formula(y,
                    climate_controls=controls_3A,
                    extra_terms=ses_bach,
                    keepincome = True,
                    controls_common=[])
res2a = run_ols(f2a, df)
quick_print(res2a, f"{y} | Model 2 (add bachelors)")
print(vif_from_formula(f2a, df).head(30))

# housing proxy
f2b = build_formula(y,
                    climate_controls=controls_3A,
                    extra_terms=ses_house,
                    keepincome = True,
                    controls_common=[])
res2b = run_ols(f2b, df)
quick_print(res2b, f"{y} | Model 2 (add housing value)")
print(vif_from_formula(f2b, df).head(30))



log_chargers_per_10k | Model 2 (add bachelors)
                             OLS Regression Results                             
Dep. Variable:     log_chargers_per_10k   R-squared:                       0.180
Model:                              OLS   Adj. R-squared:                  0.175
Method:                   Least Squares   F-statistic:                     23.03
Date:                  Fri, 30 Jan 2026   Prob (F-statistic):           3.24e-29
Time:                          23:16:09   Log-Likelihood:                -847.69
No. Observations:                  1072   AIC:                             1711.
Df Residuals:                      1064   BIC:                             1751.
Df Model:                             7                                         
Covariance Type:                    HC1                                         
                              coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------

**Model 2 (SES robustness: add one SES proxy)**
For Model 2, I’m basically asking: “If I add one more socioeconomic status dimension, do the race-share relationships fall apart — or do they stick?” When I add pct_bachelors_plus (2A), I still see negative coefficients for Black, Hispanic, and Asian share, and those are still statistically significant, which matters because it suggests the pattern isn’t only an income/SES story. In this version, pct_bachelors_plus is positive and significant, and something interesting happens with income: median household income becomes statistically significant but negative, which is a classic sign that the SES variables are stepping on each other a bit (your VIFs also reflect that). When I run the alternative version with median_housing_value (2B), I get the same overall vibe: race-share coefficients stay negative and significant, housing value is positive and significant, and income again shows up negative. So my takeaway is: Model 2 doesn’t “undo” Model 1 — it mostly reinforces that the race-share associations persist, while also showing that SES controls are highly correlated with each other and can reshuffle who “gets credit” in the model. That’s exactly why it’s a robustness check and not something I’d treat as the new baseline.

Model 3 climate robustness (3B temp-only; 3C PV-only)

In [173]:
# 3A CDD and HDD only
f3a = build_formula(y, climate_controls=controls_3A)
res3a = run_ols(f3a, df)
quick_print(res3a, f"{y} | Model 3A (HDD + CDD)")
print(vif_from_formula(f3a, df).head(30))

# 3B temp only
f3b = build_formula(y, climate_controls=controls_3B)
res3b = run_ols(f3b, df)
quick_print(res3b, f"{y} | Model 3B (temp only)")
print(vif_from_formula(f3b, df).head(30))

# 3C temp only
f3c = build_formula(y, climate_controls=controls_3C)
res3c = run_ols(f3c, df)
quick_print(res3c, f"{y} | Model 3C (GHI only)")
print(vif_from_formula(f3c, df).head(30))


log_chargers_per_10k | Model 3A (HDD + CDD)
                             OLS Regression Results                             
Dep. Variable:     log_chargers_per_10k   R-squared:                       0.177
Model:                              OLS   Adj. R-squared:                  0.172
Method:                   Least Squares   F-statistic:                     23.77
Date:                  Fri, 30 Jan 2026   Prob (F-statistic):           1.76e-26
Time:                          23:16:09   Log-Likelihood:                -849.90
No. Observations:                  1072   AIC:                             1714.
Df Residuals:                      1065   BIC:                             1749.
Df Model:                             6                                         
Covariance Type:                    HC1                                         
                              coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------

**Model 3A (HDD + CDD)**

For Model 3A, I keep the outcome as log(1 + chargers per 10,000 residents) and control for climate demand using HDD and CDD, with HC1 robust standard errors. The main pattern stays the same: higher Black, Hispanic, and Asian population shares are each associated with lower charger availability, even after controlling for income and climate demand. HDD is strongly positive, while CDD is negative but not statistically significant in this spec, which suggests heating-related demand is doing more work than cooling for explaining charger differences here.

**Model 3B (temperature-only)**

In Model 3B, I switch the climate controls to mean temperature only (t2m_mean) instead of degree days. The results are very consistent with the baseline story: Black, Hispanic, and Asian population shares remain negative and highly significant. Temperature itself is also significant, which makes sense as a broad proxy for climate and geography. This version is mainly a robustness check showing the race-share relationships don’t depend on using HDD/CDD specifically.

**Model 3C (GHI only)**

In Model 3C, I use solar resource (GHI) as the control instead of the degree-day or temperature climate measures. Even with this alternative “location/resource” control, the same core result holds: higher Black, Hispanic, and Asian shares are still associated with lower charger availability, and GHI is statistically significant as well. Because GHI is more naturally tied to solar potential than EV charger siting, I treat this as an extra robustness specification rather than the main climate-control approach for chargers.

Income is negatively associated with charger density even in the bivariate model. This is not a sign-flip artifact from controls. However, this remains an observational association and likely reflects correlated geographic/urban-form and EV-adoption factors; we treat income primarily as a control and test robustness using additional controls and alternative denominators (e.g., chargers per ZEV).

Model 4 interactions (centered)

In [174]:
df_int = center_cols(df, [income] + race)

f4 = (
    f"{y} ~ median_household_income_c + pct_black_c + pct_hispanic_c + pct_asian_c"
    + (" + " + " + ".join(["poverty_rate"] + controls_3A) if (controls_common or controls_3A) else "")
    + " + median_household_income_c:pct_black_c"
    + " + median_household_income_c:pct_hispanic_c"
    + " + median_household_income_c:pct_asian_c"
)

res4 = run_ols(f4, df_int)
quick_print(res4, f"{y} | Model 4 interactions (centered)")
print(vif_from_formula(f4, df_int).head(30))

f4R = (
    f"{y} ~ median_household_income_c + pct_black_c + pct_hispanic_c + pct_asian_c"
    + (" + " + " + ".join(controls_common + controls_3A) if (controls_common or controls_3A) else "")
    + " + median_household_income_c:pct_black_c"
    + " + median_household_income_c:pct_hispanic_c"
    + " + median_household_income_c:pct_asian_c"
)

res4R = run_ols(f4R, df_int)
quick_print(res4R, f"{y} | Model 4R interactions (centered)")
print(vif_from_formula(f4R, df_int).head(30))


log_chargers_per_10k | Model 4 interactions (centered)
                             OLS Regression Results                             
Dep. Variable:     log_chargers_per_10k   R-squared:                       0.216
Model:                              OLS   Adj. R-squared:                  0.208
Method:                   Least Squares   F-statistic:                     20.68
Date:                  Fri, 30 Jan 2026   Prob (F-statistic):           2.29e-35
Time:                          23:16:09   Log-Likelihood:                -823.95
No. Observations:                  1072   AIC:                             1670.
Df Residuals:                      1061   BIC:                             1725.
Df Model:                            10                                         
Covariance Type:                    HC1                                         
                                               coef    std err          z      P>|z|      [0.025      0.975]
-------------------------

**Model 4 (interactions, centered; no population)**

In Model 4, I add race × income interactions (with all interaction terms centered for cleaner interpretation) while keeping the same outcome, log(1 + chargers per 10,000 residents), and using HC1 robust standard errors. The main race-share coefficients stay strongly negative: higher Black, Hispanic, and Asian population shares are each associated with lower charger availability, even at average income. In this version, the interaction terms show some heterogeneity: income × Hispanic share is negative and statistically significant, while income × Asian share is positive and marginally significant. In other words, the income–charger relationship looks meaningfully different depending on ZIP racial composition in this specification.

**Model 4R (interactions, centered; adds total population)**

As a robustness check, Model 4R adds total population to the interaction model. Population is extremely predictive here (and the model fit jumps a lot), which suggests it’s capturing big structural differences across ZIPs. With population included, the main race-share coefficients remain negative and significant, but the interaction story becomes simpler: the income × Hispanic interaction is no longer significant, while the income × Asian interaction remains positive and significant. So the key takeaway is that the interaction results are somewhat sensitive to whether we control for ZIP size/urban structure, but the overall negative association between charger availability and race-share variables is consistently present across both versions.

**Model 5 utility fixed effects (+ optional clustered SEs)**

In [181]:
if utility_fe:
    f5 = build_formula(y, climate_controls=controls_3A, fe_terms=[utility_fe])

    res5 = run_ols(f5, df)
    quick_print(res5, f"{y} | Model 5 utility FE")

    res5C = run_ols(f5, df, cluster_col="county_geoid")
    quick_print(res5C, f"{y} | Model 5C clustered SEs by county")


log_chargers_per_10k | Model 5 utility FE
                             OLS Regression Results                             
Dep. Variable:     log_chargers_per_10k   R-squared:                       0.172
Model:                              OLS   Adj. R-squared:                  0.166
Method:                   Least Squares   F-statistic:                     18.43
Date:                  Fri, 30 Jan 2026   Prob (F-statistic):           6.57e-26
Time:                          23:30:31   Log-Likelihood:                -806.88
No. Observations:                  1005   AIC:                             1632.
Df Residuals:                       996   BIC:                             1676.
Df Model:                             8                                         
Covariance Type:                    HC1                                         
                                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------

**Model 5 (utility fixed effects, HC1)**

In Model 5, I add utility territory fixed effects (PG&E is the reference group, with dummy terms for SDG&E and SCE). This means I’m basically comparing ZIP codes within the same utility territory, holding constant utility-level stuff like big policy / infrastructure differences across utilities. Even with that “within-utility” comparison, the race-share coefficients stay negative and statistically significant: higher % Black, % Hispanic, and % Asian are each associated with lower log(1 + chargers per 10k). Roughly speaking, a 10 percentage point increase in (say) Hispanic share corresponds to about a 6–7% lower (1 + chargers per 10k) (since 0.1 \times -0.651 \approx -0.065). Income is also negative and significant in this spec (I’d describe that as a conditional relationship—once you hold constant race shares + climate + utility, the “leftover” variation income captures can flip signs). HDD is positive and significant; CDD is negative but only marginal here. Importantly, the utility dummy coefficients themselves aren’t significant, which suggests that once you control for the rest, SDG&E vs SCE vs PG&E differences aren’t what’s driving the within-sample variation in this outcome.

**Model 5C (same model, but clustered SEs by county)**

Model 5C is the same exact regression (so the coefficients do not change), but the standard errors are clustered by county, which is a tougher way to do inference if ZIP codes inside the same county are correlated. The main takeaway is that the story largely holds up: income and all three race-share variables are still significant, and HDD stays significant, while CDD becomes clearly not significant under clustering. So if you’re worried about “maybe I’m overstating significance because nearby ZIPs move together,” clustering by county is a solid check—and here, it mostly just makes the inference more conservative without wiping out the main patterns.

Because utility-territory identifiers are only available for PG&E, SCE, and SDG&E, Models 5 and 5C are estimated on the restricted sample of ZIP codes within these three utilities (N=1,020). As a result, the fixed-effect results should be interpreted as within-utility patterns for this subset of California ZIP codes, rather than statewide estimates.

**Model 6 geography: county FE OR lat/lon** - I think the writeup for this below needs to be corrected

In [185]:
# 6A Lat/Lon FE
f6a = build_formula(y, climate_controls=[], extra_terms=latlon)
res6a = run_ols(f6a, df)
quick_print(res6a, f"{y} | Model 6A lat and lon")
print(vif_from_formula(f6a, df).head(30))

# 6B County FE
f6b = build_formula(y, climate_controls=[], extra_terms=[county_fe])
res6b = run_ols(f6b, df)
quick_print(res6b, f"{y} | Model 6B county fe")
print(vif_from_formula(f6b, df).head(30))

res6_fe_cluster = run_ols(f1, df, cluster_col="county_geoid")
quick_print(res6_fe_cluster, f"{y} | No County FE + clustered SEs (county)")


log_chargers_per_10k | Model 6A lat and lon
                             OLS Regression Results                             
Dep. Variable:     log_chargers_per_10k   R-squared:                       0.160
Model:                              OLS   Adj. R-squared:                  0.156
Method:                   Least Squares   F-statistic:                     22.57
Date:                  Fri, 30 Jan 2026   Prob (F-statistic):           3.91e-25
Time:                          23:37:12   Log-Likelihood:                -860.55
No. Observations:                  1072   AIC:                             1735.
Df Residuals:                      1065   BIC:                             1770.
Df Model:                             6                                         
Covariance Type:                    HC1                                         
                              coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------

**Model 5 — Utility fixed effects (PGE/SCE/SDGE only)**

In Model 5, I add utility fixed effects, so the comparisons are basically happening within the same utility territory (for the ZIPs where I actually have utility labels: mostly PGE/SCE/SDGE). Even after doing that, the race-share coefficients stay strongly negative: higher Black, Hispanic, and Asian shares are each associated with lower charger availability (in log(1 + chargers per 10k)). As a rough scale check, a 10 percentage point increase in Hispanic share corresponds to about a 6.3% decrease in (1 + chargers per 10k), and a 10pp increase in Black share corresponds to about an 11% decrease. The utility dummy terms themselves aren’t significant here (so the big story isn’t “SCE vs PGE”), but rather the within-utility pattern in the demographic variables. Income is negative and significant in this restricted sample, which I’d interpret cautiously since this model is only using the subset of ZIPs covered by those three utilities.

**Model 5C — Same model, but clustered standard errors by county**

Model 5C keeps the exact same RHS, but uses county-clustered standard errors, meaning I’m being more conservative about inference because ZIPs in the same county aren’t really independent. The main pattern basically holds: the race-share coefficients stay negative and statistically significant, and income stays negative/significant. The biggest practical change is that CDD becomes less convincing (not significant once clustered), while HDD stays significant. So the “headline” from this version is: even after controlling for utility territory and being conservative about SEs, the demographic associations don’t disappear.

**Model 1R — Baseline model with county-clustered SEs (no county FE)**

For Model 1R (baseline + county-clustered SEs, but no county fixed effects), the results are clean and still very interpretable. Black, Hispanic, and Asian shares are all negative and significant, poverty rate is positive and significant, and income is not significant once clustered. In terms of magnitude, a 10 percentage point increase in Hispanic share is associated with about a 7% decrease in (1 + chargers per 10k), and a 10pp increase in Black share is about a 12% decrease. HDD is strongly positive and significant; CDD is not. So I’d feel good logging this as “done” and moving on, because it’s a solid baseline with conservative inference that you can compare all the variants against.

**Model 7 infrastructure controls**

In [191]:
# outcome -> columns to EXCLUDE from infrastructure controls
exclude_for_y = {
    "y_chargers": ["total_chargers", "level1_chargers", "level2_chargers", "dc_fast_chargers", "chargers_per_1k"],
    "y_pv": ["PV_system_size_DC", "pv_kw_per_1k"],
    "y_storage": ["storage_capacity_mw", "storage_mw_per_100k"],
    "y_wind_mw": ["wind_capacity_mw", "wind_mw_per_100k"],
    "y_turbines": ["wind_turbine_count"],
}

# candidate infrastructure controls (RAW)
infra_candidates = ["plant_capacity_mw", "storage_capacity_mw", "wind_capacity_mw", "wind_turbine_count", "PV_system_size_DC"]

infra = [c for c in infra_candidates if c in df.columns]
infra = [c for c in infra if c not in exclude_for_y.get(y, [])]

f7 = build_formula(y, climate_controls=controls_3A, extra_terms=infra)
res7 = run_ols(f7, df)
quick_print(res7, f"{y} | Model 7 (infrastructure controls, outcome-safe)")
print(vif_from_formula(f7, df).head(30))


log_chargers_per_10k | Model 7 (infrastructure controls, outcome-safe)
                             OLS Regression Results                             
Dep. Variable:     log_chargers_per_10k   R-squared:                       0.289
Model:                              OLS   Adj. R-squared:                  0.282
Method:                   Least Squares   F-statistic:                     26.42
Date:                  Fri, 30 Jan 2026   Prob (F-statistic):           8.66e-49
Time:                          23:53:18   Log-Likelihood:                -722.58
No. Observations:                  1046   AIC:                             1469.
Df Residuals:                      1034   BIC:                             1529.
Df Model:                            11                                         
Covariance Type:                    HC1                                         
                              coef    std err          z      P>|z|      [0.025      0.975]
--------------------------

In [194]:
# Create per-capita infrastructure controls once
pop = df["total_population"].replace(0, np.nan)

if "plant_capacity_mw" in df.columns:
    df["plant_mw_per_100k"] = df["plant_capacity_mw"] * 100000 / pop
if "storage_capacity_mw" in df.columns:
    df["storage_mw_per_100k"] = df["storage_capacity_mw"] * 100000 / pop
if "wind_capacity_mw" in df.columns:
    df["wind_mw_per_100k_ctrl"] = df["wind_capacity_mw"] * 100000 / pop
if "wind_turbine_count" in df.columns:
    df["turbines_per_100k"] = df["wind_turbine_count"] * 100000 / pop

infra_pc = [c for c in ["plant_mw_per_100k", "storage_mw_per_100k", "wind_mw_per_100k_ctrl"] if c in df.columns]
f7pc = build_formula(y, climate_controls=controls_3A, extra_terms=infra_pc)
res7pc = run_ols(f7pc, df)
quick_print(res7pc, f"{y} | Model 7 (per-capita infrastructure controls)")
print(vif_from_formula(f7pc, df).head(30))


log_chargers_per_10k | Model 7 (per-capita infrastructure controls)
                             OLS Regression Results                             
Dep. Variable:     log_chargers_per_10k   R-squared:                       0.396
Model:                              OLS   Adj. R-squared:                  0.391
Method:                   Least Squares   F-statistic:                     60.83
Date:                  Fri, 30 Jan 2026   Prob (F-statistic):           2.44e-89
Time:                          23:55:11   Log-Likelihood:                -637.62
No. Observations:                  1046   AIC:                             1295.
Df Residuals:                      1036   BIC:                             1345.
Df Model:                             9                                         
Covariance Type:                    HC1                                         
                              coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------

In Model 7, I add per-capita DER-intensity controls (distributed generation and storage capacity per 100k residents) to capture differences in local clean-energy buildout that could be correlated with EV charging availability. These DER measures are strongly associated with the outcome, and the model fit increases substantially. Even with these controls, higher Black, Hispanic, and Asian population shares remain negatively associated with charger availability. Median income becomes negative and significant in this specification, which I interpret as a conditional effect once DER buildout and other correlated factors are held fixed, rather than a standalone income–charger relationship.

**Model 8 demand proxy robustness**

In [195]:
if demand_proxy:
    f8 = build_formula(y, climate_controls=controls_3A, extra_terms=[demand_proxy])
    res8 = run_ols(f8, df)
    quick_print(res8, f"{y} | Model 8 add demand proxy")
    print(vif_from_formula(f8, df).head(30))


log_chargers_per_10k | Model 8 add demand proxy
                             OLS Regression Results                             
Dep. Variable:     log_chargers_per_10k   R-squared:                       0.181
Model:                              OLS   Adj. R-squared:                  0.175
Method:                   Least Squares   F-statistic:                     22.39
Date:                  Fri, 30 Jan 2026   Prob (F-statistic):           3.39e-28
Time:                          23:55:38   Log-Likelihood:                -777.96
No. Observations:                   972   AIC:                             1572.
Df Residuals:                       964   BIC:                             1611.
Df Model:                             7                                         
Covariance Type:                    HC1                                         
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------

In Model 8, I add a demand proxy (log_kwh), measured as utility-reported electricity demand by ZIP code for PG&E, SCE, and SDG&E service territories. Using OLS with HC1 robust standard errors, I find log_kwh is negative and statistically significant, indicating that higher electricity demand is associated with slightly lower charger availability per 10,000 residents after controlling for income, demographic composition, and HDD/CDD. The negative coefficients on Black, Hispanic, and Asian population shares remain large and statistically significant, suggesting the main distributional pattern is not explained away by differences in local electricity demand.